In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import re

def limpar_texto(texto):
    if not isinstance(texto, str):
        texto = str(texto)
    texto = texto.lower()
    texto = re.sub(r'\d+', '', texto) # Remove números
    texto = re.sub(r'[^\w\s]', ' ', texto) # Remove pontuação
    texto = " ".join(texto.split()) # Remove espaços extras
    return texto

path = r'C:\Users\Lucas Martins\Carmel Capital\Arquivos - Documentos\00 - CARMEL ASSET\01 - OPERACIONAL\CONTROLADORIA\01 - Relatorios Diarios\Caixa Diario\DADOS_TREINAMENTO\gerar.xlsx'
df_historico = pd.read_excel(path, sheet_name='dicionario')

# 1. TRATAMENTO DOS DADOS
df_historico['Historico_Limpo'] = df_historico['Historico'].apply(limpar_texto)

# Garante que 'y' seja string e remove possíveis espaços em branco nas categorias
y = df_historico['Controladoria'].fillna('classificar').astype(str).str.strip()

X = df_historico['Historico_Limpo']
# --- NOVO TRECHO PARA CORRIGIR O ERRO ---
# Conta quantos exemplos existem de cada categoria
contagem_classes = y.value_counts()
# Identifica as classes que têm apenas 1 exemplo (o que quebra o stratify)
classes_invalidas = contagem_classes[contagem_classes < 2].index
if len(classes_invalidas) > 0:
    print(f"Aviso: Foram removidas as seguintes classes por terem apenas 1 exemplo: {list(classes_invalidas)}")
# Filtra o X e o y mantendo apenas as classes válidas (com 2 ou mais exemplos)
mascara = ~y.isin(classes_invalidas)
X = X[mascara]
y = y[mascara]
# ----------------------------------------
# 2. SEPARANDO TREINO E TESTE
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# 3. CRIANDO E TREINANDO O MODELO
modelo = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2))),
    ('classificador', LinearSVC(class_weight='balanced', random_state=42))
])

modelo.fit(X_treino, y_treino)

# 4. TESTANDO E VALIDANDO
previsoes = modelo.predict(X_teste)

print(f"Taxa de Acerto (Acurácia): {accuracy_score(y_teste, previsoes) * 100:.2f}%\n")
print("--- RELATÓRIO DE CLASSIFICAÇÃO ---")
print(classification_report(y_teste, previsoes))

# 5. MATRIZ DE CONFUSÃO
labels_modelo = modelo.classes_
matriz = confusion_matrix(y_teste, previsoes, labels=labels_modelo)

df_matriz = pd.DataFrame(
    matriz, 
    index=[f"Real: {c}" for c in labels_modelo], 
    columns=[f"Previsto: {c}" for c in labels_modelo]
)

print("--- MATRIZ DE CONFUSÃO ---")
print(df_matriz)

Aviso: Foram removidas as seguintes classes por terem apenas 1 exemplo: ['classificar']
Taxa de Acerto (Acurácia): 99.47%

--- RELATÓRIO DE CLASSIFICAÇÃO ---
                                        precision    recall  f1-score   support

                                     0       1.00      1.00      1.00         2
Aplicação de Fundos - Zeragem de Caixa       1.00      0.99      1.00       132
            Aporte de Cotas - Mezanino       1.00      1.00      1.00         7
                  Cessão de Recebíveis       0.98      1.00      0.99       167
                   IOF Aporte de Cotas       1.00      1.00      1.00         5
                 Liquidação de Títulos       1.00      0.88      0.94        17
           Resgate de Cotas - Mezanino       1.00      1.00      1.00         2
  Resgate de Fundos - Zeragem de Caixa       1.00      1.00      1.00       174
                         Saldo Inicial       1.00      1.00      1.00         2
                           Taxa ANBIMA   